# Assignment 4: Retrieval-augmented generation

Build a RAG pipeline over the [PubMedQA](https://github.com/pubmedqa/pubmedqa) medical question-answering dataset using **LangChain**.

Architecture: question → embed → vector store retrieval → augment LM prompt → answer yes/no.

We use **Option B** (LCEL chain with `RunnableParallel`) to keep both the retrieved context and the generated answer accessible for evaluation.


## Setup

Install LangChain + dependencies. In Colab this takes ~1 minute.


In [1]:
!pip install -q langchain langchain-community langchain-huggingface langchain-core langchain-chroma sentence-transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 73.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 96.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 85.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

### Configuration

Choose the generative LM here. The pipeline works with several options:

- `Qwen/Qwen2.5-1.5B-Instruct` — open access, comparable quality, no waiting
- `HuggingFaceTB/SmolLM2-360M-Instruct` — smallest/fastest fallback


In [2]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
# MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

import torch
DEVICE = 0 if torch.cuda.is_available() else -1   # HF pipeline convention: device int
print(f"Generator: {MODEL_NAME}")
print(f"Embedder:  {EMBEDDING_MODEL}")
print(f"Device:    {'cuda' if DEVICE == 0 else 'cpu'}")


Generator: Qwen/Qwen2.5-1.5B-Instruct
Embedder:  sentence-transformers/all-MiniLM-L6-v2
Device:    cuda


## Part 1: The dataset

### ⚙ Task 1.1 — Downloading and inspecting the QA dataset

PubMedQA: medical research questions paired with abstracts and yes/no answers.

> **⚙ Notes — Task 1.1**
> - Each row: question, context paragraphs, long answer, year, yes/no/maybe label
> - We keep only **yes/no** rows (drop "maybe") → cleanly binary classification
> - Two derived tables:
>   - `documents`: abstract + long_answer concatenated (the corpus to retrieve over)
>   - `questions`: question + gold label + gold document id (for evaluation)


In [3]:
!wget -q https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json


In [4]:
import pandas as pd

tmp_data = pd.read_json("ori_pqal.json").T
# Only keep yes/no — drop "maybe" so the labels are binary.
tmp_data = tmp_data[tmp_data.final_decision.isin(["yes", "no"])]

documents = pd.DataFrame({
    "abstract": tmp_data.apply(lambda row: " ".join(row.CONTEXTS + [row.LONG_ANSWER]), axis=1),
    "year": tmp_data.YEAR,
})
questions = pd.DataFrame({
    "question": tmp_data.QUESTION,
    "year": tmp_data.YEAR,
    "gold_label": tmp_data.final_decision,
    "gold_context": tmp_data.LONG_ANSWER,
    "gold_document_id": documents.index,
})

print(f"# documents: {len(documents)}")
print(f"# questions: {len(questions)}")
print(f"label counts:")
print(questions.gold_label.value_counts())


# documents: 890
# questions: 890
label counts:
gold_label
yes    552
no     338
Name: count, dtype: int64


**Sanity check** — peek at one question and one document.

In [5]:
print("=== Example question ===")
print(questions.iloc[0].question)
print(f"gold label: {questions.iloc[0].gold_label}")
print(f"\n=== Corresponding document (first 500 chars) ===")
print(documents.iloc[0].abstract[:500] + "...")


=== Example question ===
Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
gold label: yes

=== Corresponding document (first 500 chars) ===
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has b...


## Part 2: Configure the LangChain LM

### ⚙ Task 2.1 — Select a language model

> **⚙ Notes — Task 2.1**
> - **Final choice**: `Qwen/Qwen2.5-1.5B-Instruct` (Llama-3.2 gated access was rejected)
> - Replaced `HuggingFacePipeline` with a custom `RunnableLambda` that applies Qwen's **ChatML chat template** manually. Without this, Qwen ignores `<|im_end|>` and rambles past the answer.
> - `do_sample=False` for **deterministic, reproducible** evaluation
> - `eos_token_id=[tokenizer.eos_token_id, im_end_id]` ensures generation stops at end-of-turn
> - **Why instruction-tuned?** Base models continue text statistically; instruction-tuned ones answer questions. RAG needs the latter.
> - **Why 1.5B?** Larger = better quality, but T4 (16GB) tops out around 7B in fp16. 1.5B in fp16 ≈ 3GB, leaves plenty of room for the embeddings + Chroma index.


In [10]:
from langchain_core.runnables import RunnableLambda
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Load model + tokenizer manually (so we can apply the chat template ourselves)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="cuda" if torch.cuda.is_available() else "cpu",
)

# Get the proper stop token id for Qwen's <|im_end|>
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

def chat_invoke(prompt_text):
    """Take any string prompt, wrap in Qwen's ChatML, generate, return reply only."""
    # If LangChain passed a ChatPromptValue, convert to string first
    if hasattr(prompt_text, "to_string"):
        prompt_text = prompt_text.to_string()

    # Wrap as a single user message and apply the chat template
    messages = [{"role": "user", "content": str(prompt_text)}]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            eos_token_id=[tokenizer.eos_token_id, im_end_id],
            pad_token_id=tokenizer.eos_token_id,
        )
    # Decode only the newly generated tokens (skip the prompt portion)
    gen_ids = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


llm = RunnableLambda(chat_invoke)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

**Sanity check** — make sure the LLM responds to a simple prompt.

In [11]:
out = llm.invoke("What is the capital of Sweden? Answer in one sentence.")
print(out)


The capital of Sweden is Stockholm.


## Part 3: Set up the document database

### 🎓 Task 3.1 — Embedding model

> **🎓 Notes — Task 3.1**
> - **Choice**: `all-MiniLM-L6-v2` (22M params, 384-dim)
> - Output dim verified: **384** ✓
> - **`normalize_embeddings=True`** → cosine similarity becomes equivalent to dot product (faster) and Chroma can use HNSW more efficiently
> - **Why this embedder vs a medical one?**
>   - MiniLM is fast (~3000 sentences/sec on T4) and general-purpose
>   - Medical-domain encoders (PubMedBERT-based) sometimes do marginally better on PubMedQA but are slower
>   - Our retrieval accuracy ends up at **97% top-1** with MiniLM, so domain-specific isn't necessary
> - **Why pretrained, not trained-on-task?** Training embeddings requires labeled query-document pairs we don't have. Pretrained sentence embeddings are zero-shot — that's the point of sentence-transformers.
> - **Token limit**: MiniLM truncates at 256 tokens, hence the need for chunking


In [12]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True},  # normalize → cosine sim = dot product
)

# Sanity check: encode one query, look at the shape.
v = embeddings.embed_query("What is programmed cell death?")
print(f"Embedding dim: {len(v)} (expected 384 for MiniLM-L6-v2)")
print(f"First 5 dims:  {v[:5]}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dim: 384 (expected 384 for MiniLM-L6-v2)
First 5 dims:  [-0.038830287754535675, 0.005878914147615433, -0.07347593456506729, -0.01866328902542591, 0.020866967737674713]


### ⚙ Task 3.2 — Chunking

> **⚙ Notes — Task 3.2**
> - **Splitter**: `RecursiveCharacterTextSplitter` — splits on paragraph → sentence → word → char in that order
> - **Default**: chunk_size=500, overlap=50 → **3510 chunks from 890 documents** (avg 3.94 chunks/doc)
> - Preserved `id` in metadata for downstream retrieval evaluation
> - **Reflection: how does chunking affect RAG quality?** Tested empirically in Part 5 chunking sweep:
>   - **chunk_size=300** (small): 5646 chunks, accuracy 0.56
>   - **chunk_size=500** (medium): 3510 chunks, accuracy 0.54
>   - **chunk_size=1000** (large): **1902 chunks, accuracy 0.68** ← winner
>   - The PubMedQA abstracts are information-dense — giving the LM **more context per retrieval** matters more than precise small fragments. Small chunks fragment evidence; the relevant sentence and its surrounding context end up in different chunks.
>   - For sparser corpora (e.g. FAQs, conversational data), the trade-off would flip — smaller chunks would win.


In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

# Keep the document id in metadata so we can check retrieval accuracy later.
metadatas = [{"id": idx} for idx in documents.index]
texts = text_splitter.create_documents(
    texts=documents.abstract.tolist(),
    metadatas=metadatas,
)
print(f"Original documents: {len(documents)}")
print(f"After chunking:     {len(texts)} chunks")
print(f"Avg chunks/doc:     {len(texts) / len(documents):.2f}")


Original documents: 890
After chunking:     3510 chunks
Avg chunks/doc:     3.94


**Sanity check** — look at a few chunks to confirm they're sensible.

In [14]:
for i in [0, 1, 100]:
    print(f"--- chunk {i} (id={texts[i].metadata['id']}, {len(texts[i].page_content)} chars) ---")
    print(texts[i].page_content[:300] + ("..." if len(texts[i].page_content) > 300 else ""))
    print()


--- chunk 0 (id=21645374, 498 chars) ---
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cel...

--- chunk 1 (id=21645374, 496 chars) ---
has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided int...

--- chunk 100 (id=23690198, 497 chars) ---
This audit cycle measured epidural analgesia performance against 4 standards: (1) Implementation of epidural analgesia for labor to all patients; (2) Acceptance and good satisfaction level reported by patients and caregivers. (3) Effectiveness of labor analgesia; (

### 🎓 Task 3.3 — Vector store (Chroma)

> **🎓 Notes — Task 3.3**
> - **Chroma**: lightweight in-memory vector DB, native LangChain integration
> - **Cosine similarity** explicitly set via `collection_metadata={"hnsw:space": "cosine"}`
> - Under the hood: **HNSW** (Hierarchical Navigable Small World) graph for approximate nearest neighbors → **O(log n) lookup** rather than O(n) linear scan
> - At insert time: each chunk → embedding → stored together with metadata
> - At query time: query → embedding → top-k chunks ranked by cosine distance
> - **Sanity check** with "What is programmed cell death?" returned the correct top-1 doc (the lace plant PCD abstract) with similarity 0.381 — much lower (better, in Chroma's cosine-distance convention) than the next two
> - **Why vector DB vs just numpy?** 1000 docs is fine in numpy, but real RAG corpora have millions of docs. HNSW scales sub-linearly with corpus size.


In [15]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    collection_metadata={"hnsw:space": "cosine"},
)
print(f"Indexed {vector_store._collection.count()} chunks")


Indexed 3510 chunks


**Sanity check** — top-3 retrieval for a question that should match the corpus.

In [16]:
results = vector_store.similarity_search_with_score(
    "What is programmed cell death?", k=3,
)
for res, score in results:
    print(f"* [SIM={score:.3f}] id={res.metadata['id']}")
    print(f"  {res.page_content[:200]}...")
    print()


* [SIM=0.381] id=21645374
  Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant co...

* [SIM=0.604] id=15223779
  mutations in exon 11 of the KIT gene were found. On the contrary, expression of the stem cell growth factor (c-kit ligand) was detected in all three uveal melanoma cell lines, suggesting the presence ...

* [SIM=0.635] id=18158048
  syndrome can harbor viable germ cell elements or seminiferous tubules. The exact fate of these residual elements remains unknown; however, there may exist the potential for malignant transformation. G...



## Part 4: Implementing the RAG pipeline (Option B: LCEL chain)

### 🎓 Task 4.1 — Defining the full RAG pipeline

> **🎓 Notes — Task 4.1 (Option B with LCEL)**
> 
> **Architecture** (built with LangChain Expression Language):
> 1. **`RunnableParallel`** runs two computations on the same input:
>    - branch 1: question → `retriever` → list of `Document`s → format as context string
>    - branch 2: question → `RunnablePassthrough` (unchanged)
> 2. Both feed into the prompt template, then LLM, then `StrOutputParser`
> 3. **`.assign(answer=chain)`** attaches the LLM output back into the parallel dict → output has BOTH `context`/`docs` AND `answer` for downstream evaluation
> 
> **Two chain variants built**:
> - `rag_chain`: returns `{context, question, answer}` — context is a string
> - `rag_chain_with_docs`: returns `{docs, question, answer}` — docs are raw `Document` objects with `.metadata['id']` for retrieval accuracy evaluation (Task 5.2)
> 
> **Prompt design**: Explicit instruction "answer MUST start with Yes or No" — makes downstream parsing robust. Worked perfectly: **valid_rate = 1.000** on 100 questions (zero parse failures).
> 
> **Sanity check** on the PCD lace plant question: retrieved the correct gold doc (id=21645374), answered "Yes" with a reasonable explanation referencing the context ✓


In [17]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Number of chunks to retrieve per question. Start with 1 (assignment hint).
TOP_K = 1
retriever = vector_store.as_retriever(search_kwargs={"k": TOP_K})

# Prompt template. The instruction "answer with Yes or No" makes the model's
# output much easier to parse downstream.
prompt = ChatPromptTemplate.from_template(
    """You are a medical assistant. Use the following retrieved context to answer the question.
The answer MUST start with either "Yes" or "No", followed by a short explanation.

Context:
{context}

Question: {question}

Answer:"""
)


def format_docs(docs):
    """Concatenate retrieved chunks into one context string."""
    return "\n\n".join(d.page_content for d in docs)


# Step 1: parallel — get context and pass through the question.
parallel = RunnableParallel(
    context=retriever | format_docs,
    question=RunnablePassthrough(),
)

# Step 2: the chain that consumes context+question and produces an answer string.
generation_chain = prompt | llm | StrOutputParser()

# Step 3: combine. .assign() adds the chain output as a new key in the dict,
# WITHOUT dropping the existing keys — so we get {context, question, answer}.
rag_chain = parallel.assign(answer=generation_chain)

# Also build a version that returns the raw retrieved Documents (with metadata)
# alongside the answer — needed for Task 5.2 (retrieval accuracy).
rag_chain_with_docs = RunnableParallel(
    docs=retriever,
    question=RunnablePassthrough(),
).assign(
    answer=(
        RunnableParallel(
            context=lambda x: format_docs(x["docs"]),
            question=lambda x: x["question"],
        )
        | prompt
        | llm
        | StrOutputParser()
    )
)


**Sanity check** — run on a question from the dataset and inspect both retrieved doc and answer.

In [18]:
q = questions.iloc[0].question
print(f"QUESTION: {q}")
print(f"GOLD: {questions.iloc[0].gold_label}")
print()

out = rag_chain_with_docs.invoke(q)
print("RETRIEVED:")
for d in out["docs"]:
    print(f"  id={d.metadata['id']}, {len(d.page_content)} chars")
    print(f"  {d.page_content[:200]}...")
print()
print(f"ANSWER: {out['answer']}")


QUESTION: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
GOLD: yes

RETRIEVED:
  id=21645374, 498 chars
  Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant co...

ANSWER: Yes

Explanation: The provided context mentions that the role of mitochondria during programmed cell death (PCD) has been recognized in animals. Lace plants produce perforations in their leaves through programmed cell death, which aligns with the concept of programmed cell death observed in other organisms like animals. Therefore, it can be inferred that mitochondria likely play a role in remodelling lace plant leaves during this process.


## Part 5: Evaluate RAG on the dataset

### 🎓 Task 5.1 — High-level evaluation

> **🎓 Notes — Task 5.1**
> 
> **Setup**: 100 test questions, deterministic generation (`do_sample=False`), parse Yes/No from start of answer
> 
> **My results**:
> 
> | Setup | accuracy | F1 (pos=yes) | valid_rate |
> |---|---|---|---|
> | **RAG (k=1, chunk=500)** | **0.580** | **0.708** | 1.000 |
> | Baseline (no retrieval) | 0.440 | 0.429 | 1.000 |
> | **Improvement from RAG** | **+14 pts** | **+28 pts** | — |
> 
> **Key takeaways**:
> - **Retrieval clearly helps** — the +14 accuracy / +28 F1 gap is substantial and well above noise (sample size 100)
> - **Both setups have 100% valid_rate** — the "answer MUST start with Yes or No" instruction is robust; Qwen always complies
> - **The dataset is imbalanced** (552 yes vs 338 no, ~62% yes). The baseline F1 of 0.429 means it's not even predicting "yes" naively (which would give F1 ≈ 0.78). Looking at baseline outputs: the LM tries to genuinely reason from prior knowledge, often defaults to "No" when uncertain.
> - **RAG's F1 (0.71) >> baseline F1 (0.43)** — retrieval gives the model concrete evidence to find the right answer in


In [19]:
import re
from sklearn.metrics import f1_score, accuracy_score
from tqdm.auto import tqdm

# How many test questions to evaluate on. Use all for the final report;
# a smaller subset (e.g. 50) is fine while iterating.
N_EVAL = 100   # set to len(questions) for full eval (will take longer)


def parse_yes_no(text):
    """Return 'yes', 'no', or None depending on what the model said first."""
    m = re.match(r"\s*(yes|no)\b", text, re.IGNORECASE)
    return m.group(1).lower() if m else None


def evaluate(predict_fn, questions_df, n=N_EVAL):
    """Run predict_fn on each question, collect yes/no, return metrics + per-row info."""
    rows = []
    for _, q in tqdm(list(questions_df.head(n).iterrows()), total=n):
        try:
            raw = predict_fn(q.question)
        except Exception as e:
            raw = f"<error: {e}>"
        rows.append({
            "question": q.question,
            "gold": q.gold_label,
            "gold_doc_id": q.gold_document_id,
            "raw": raw["answer"] if isinstance(raw, dict) else raw,
            "retrieved_doc_ids": [d.metadata["id"] for d in raw["docs"]] if isinstance(raw, dict) and "docs" in raw else [],
        })
    df = pd.DataFrame(rows)
    df["pred"] = df["raw"].apply(parse_yes_no)

    # Compute metrics only on rows where the model gave a valid yes/no.
    valid = df[df["pred"].notna()]
    n_valid = len(valid)
    if n_valid == 0:
        return {"accuracy": 0.0, "f1": 0.0, "valid_rate": 0.0}, df
    acc = accuracy_score(valid["gold"], valid["pred"])
    f1 = f1_score(valid["gold"], valid["pred"], pos_label="yes")
    return {
        "accuracy": acc,
        "f1": f1,
        "valid_rate": n_valid / len(df),
        "n_evaluated": len(df),
        "n_valid": n_valid,
    }, df


In [20]:
# --- Evaluate RAG pipeline ---
print("Evaluating RAG pipeline...")
rag_metrics, rag_df = evaluate(
    lambda q: rag_chain_with_docs.invoke(q),
    questions,
)
print("\nRAG metrics:")
for k, v in rag_metrics.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")


Evaluating RAG pipeline...


  0%|          | 0/100 [00:00<?, ?it/s]


RAG metrics:
  accuracy: 0.580
  f1: 0.708
  valid_rate: 1.000
  n_evaluated: 100
  n_valid: 100


In [21]:
# --- Baseline: same LM, no retrieval ---
baseline_prompt = ChatPromptTemplate.from_template(
    """You are a medical assistant. Answer the following question.
The answer MUST start with either "Yes" or "No", followed by a short explanation.

Question: {question}

Answer:"""
)
baseline_chain = baseline_prompt | llm | StrOutputParser()

print("\nEvaluating baseline (no retrieval)...")
baseline_metrics, baseline_df = evaluate(
    lambda q: baseline_chain.invoke({"question": q}),
    questions,
)
print("\nBaseline metrics:")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")



Evaluating baseline (no retrieval)...


  0%|          | 0/100 [00:00<?, ?it/s]


Baseline metrics:
  accuracy: 0.440
  f1: 0.429
  valid_rate: 1.000
  n_evaluated: 100
  n_valid: 100


In [22]:
# --- Side-by-side ---
print("="*60)
print(f'{"Setup":<25s} {"acc":>8s} {"f1":>8s} {"valid":>8s}')
print("-"*60)
print(f'{"RAG (k=" + str(TOP_K) + ", chunk=" + str(CHUNK_SIZE) + ")":<25s} '
      f'{rag_metrics["accuracy"]:>8.3f} {rag_metrics["f1"]:>8.3f} {rag_metrics["valid_rate"]:>8.3f}')
print(f'{"Baseline (no retrieval)":<25s} '
      f'{baseline_metrics["accuracy"]:>8.3f} {baseline_metrics["f1"]:>8.3f} {baseline_metrics["valid_rate"]:>8.3f}')


Setup                          acc       f1    valid
------------------------------------------------------------
RAG (k=1, chunk=500)         0.580    0.708    1.000
Baseline (no retrieval)      0.440    0.429    1.000


### 🎓 Task 5.1 (extra) — Chunking parameter sweep

> **🎓 Notes — chunking sweep**
> 
> Tested 4 configurations on 50 questions each:
> 
> | chunk_size | overlap | top_k | # chunks | accuracy | F1 |
> |---|---|---|---|---|---|
> | 300 | 30 | 1 | 5646 | 0.56 | 0.676 |
> | 500 | 50 | 1 | 3510 | 0.54 | 0.676 |
> | **1000** | **100** | **1** | **1902** | **0.68** | **0.778** |
> | 500 | 50 | 3 | 3510 | 0.60 | 0.714 |
> 
> **Observations**:
> 1. **Bigger chunks (1000) clearly win on PubMedQA** — accuracy jumped from 0.54 → 0.68 (+14 pts). Information-dense scientific abstracts benefit from more contiguous context.
> 2. **Top-3 retrieval (last row) helps marginally over top-1** at same chunk size (0.60 vs 0.54). Trade-off: more context → easier for the LM to find the answer, but also more noise.
> 3. **Top-3 with chunk_size=500 (0.60) is worse than top-1 with chunk_size=1000 (0.68)** — same total context length (~1500 chars) but the larger contiguous chunk wins. Fragmentation across multiple chunks hurts.
> 4. **chunk_size=300 ≈ chunk_size=500** — diminishing returns from going smaller; in fact 300 is marginally better than 500 here (noise on 50 samples).
> 
> **Why didn't I rerun the main evaluation with chunk_size=1000?** I committed to chunk=500 as my "default" configuration earlier. The sweep itself is the experiment. In a real project I'd lock in chunk=1000 going forward.


In [23]:
# Reduced for the sweep — bump up after picking the best config.
N_EVAL_SWEEP = 50

sweep_configs = [
    # (chunk_size, chunk_overlap, top_k)
    (300, 30, 1),
    (500, 50, 1),
    (1000, 100, 1),
    (500, 50, 3),    # same chunking, more retrieved docs
]

sweep_results = []
for cs, co, k in sweep_configs:
    print(f"\n--- chunk_size={cs}, overlap={co}, top_k={k} ---")
    # Re-chunk
    splitter = RecursiveCharacterTextSplitter(chunk_size=cs, chunk_overlap=co)
    chunks = splitter.create_documents(texts=documents.abstract.tolist(), metadatas=metadatas)
    # Re-index (new collection per setting, otherwise they'd be mixed)
    vs = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_metadata={"hnsw:space": "cosine"},
        collection_name=f"sweep_cs{cs}_co{co}",
    )
    r = vs.as_retriever(search_kwargs={"k": k})

    # Build a temporary chain
    chain_with_docs = RunnableParallel(
        docs=r,
        question=RunnablePassthrough(),
    ).assign(
        answer=(
            RunnableParallel(
                context=lambda x: format_docs(x["docs"]),
                question=lambda x: x["question"],
            )
            | prompt
            | llm
            | StrOutputParser()
        )
    )

    metrics, _ = evaluate(
        lambda q: chain_with_docs.invoke(q),
        questions,
        n=N_EVAL_SWEEP,
    )
    metrics.update({"chunk_size": cs, "overlap": co, "top_k": k, "n_chunks": len(chunks)})
    sweep_results.append(metrics)

sweep_df = pd.DataFrame(sweep_results)
print("\n=== Sweep results ===")
print(sweep_df[["chunk_size", "overlap", "top_k", "n_chunks", "accuracy", "f1", "valid_rate"]].to_string(index=False))



--- chunk_size=300, overlap=30, top_k=1 ---


  0%|          | 0/50 [00:00<?, ?it/s]


--- chunk_size=500, overlap=50, top_k=1 ---


  0%|          | 0/50 [00:00<?, ?it/s]


--- chunk_size=1000, overlap=100, top_k=1 ---


  0%|          | 0/50 [00:00<?, ?it/s]


--- chunk_size=500, overlap=50, top_k=3 ---


  0%|          | 0/50 [00:00<?, ?it/s]


=== Sweep results ===
 chunk_size  overlap  top_k  n_chunks  accuracy       f1  valid_rate
        300       30      1      5646      0.56 0.676471         1.0
        500       50      1      3510      0.54 0.676056         1.0
       1000      100      1      1902      0.68 0.777778         1.0
        500       50      3      3510      0.60 0.714286         1.0


### 🎓 Task 5.2 — Detailed inspection

> **🎓 Notes — Task 5.2**
> 
> **Retrieval accuracy (gold doc in top-1)**: **97%** (97/100)
> - Retrieval is **NOT the bottleneck** — MiniLM-L6-v2 finds the right document almost every time
> - The 3 retrieval misses are likely cases where the question paraphrases the abstract heavily, or the question is ambiguous
> 
> **Quadrant breakdown** (`retrieval_hit × answer_correct`):
> 
> | | answer correct | answer wrong | total |
> |---|---|---|---|
> | retrieved gold doc | **58** | **39** | 97 |
> | missed gold doc | 0 | 3 | 3 |
> | **total** | **58** | **42** | **100** |
> 
> **The real bottleneck is reading comprehension** — of 97 questions where retrieval succeeded, the LM still got **39 wrong** (40% error rate even with the right document in context). This is where a bigger or fine-tuned LM would help most.
> 
> **Zero "lucky" cases** (wrong doc → right answer): confirms the model is genuinely using retrieved context, not relying on prior knowledge
> 
> **Concrete failure mode example (right doc + wrong answer)**:
> - Q: *"Can tailored interventions increase mammography use among HMO women?"* (gold: yes)
> - Retrieved: correct gold doc
> - Model said: "No. The provided context does not mention anything about..." 
> - **Problem**: model misread the context — the abstract does discuss tailored interventions increasing mammography use, but the model hallucinated that it didn't. Classic small-LLM reading-comp failure.
> 
> **What would improve quality?**
> - **Bigger LM**: Llama-3.1-8B or Qwen2.5-7B would likely halve the reading-comp errors
> - **Better prompting**: chain-of-thought ("First identify the relevant sentence, then answer"), few-shot examples
> - **Larger chunks**: confirmed above — chunk_size=1000 already added +14 accuracy points
> - **Reranking**: retrieve top-5, rerank with a cross-encoder, pass top-1 to LM


In [24]:
# Retrieval accuracy on the RAG df we already collected.
rag_df["retrieval_hit"] = rag_df.apply(
    lambda r: r["gold_doc_id"] in r["retrieved_doc_ids"], axis=1
)
retrieval_acc = rag_df["retrieval_hit"].mean()
print(f"Retrieval accuracy (gold doc in top-{TOP_K}): {retrieval_acc:.3f}")
print(f"  on {len(rag_df)} questions, top-{TOP_K} retrieved")


Retrieval accuracy (gold doc in top-1): 0.970
  on 100 questions, top-1 retrieved


In [25]:
# Quadrant breakdown: retrieval hit vs answer correctness
rag_df["answer_correct"] = (rag_df["pred"] == rag_df["gold"])
crosstab = pd.crosstab(
    rag_df["retrieval_hit"].map({True: "retrieved gold doc", False: "missed gold doc"}),
    rag_df["answer_correct"].map({True: "answer correct", False: "answer wrong"}),
    margins=True,
)
print(crosstab)


answer_correct      answer correct  answer wrong  All
retrieval_hit                                        
missed gold doc                  0             3    3
retrieved gold doc              58            39   97
All                             58            42  100


In [26]:
# Look at a few cases: 2 successes, 2 retrieval misses, 2 reading-comp fails.
def show_examples(df, mask, label, n=2):
    print(f"\n=== {label} ({mask.sum()} cases, showing {n}) ===")
    for _, r in df[mask].head(n).iterrows():
        print(f"Q: {r['question']}")
        print(f"GOLD: {r['gold']}   PRED: {r['pred']}")
        print(f"Gold doc id: {r['gold_doc_id']}, retrieved: {r['retrieved_doc_ids']}")
        print(f"Model said: {r['raw'][:200]}")
        print()

show_examples(rag_df, rag_df["retrieval_hit"] & rag_df["answer_correct"], "RIGHT DOC + RIGHT ANSWER")
show_examples(rag_df, rag_df["retrieval_hit"] & ~rag_df["answer_correct"], "RIGHT DOC + WRONG ANSWER (LM reading fail)")
show_examples(rag_df, ~rag_df["retrieval_hit"] & ~rag_df["answer_correct"], "WRONG DOC + WRONG ANSWER (retrieval fail)")
show_examples(rag_df, ~rag_df["retrieval_hit"] & rag_df["answer_correct"], "WRONG DOC + RIGHT ANSWER (lucky/prior)")



=== RIGHT DOC + RIGHT ANSWER (58 cases, showing 2) ===
Q: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
GOLD: yes   PRED: yes
Gold doc id: 21645374, retrieved: [21645374]
Model said: Yes

Explanation: The provided context mentions that the role of mitochondria during programmed cell death (PCD) has been recognized in animals. Lace plants produce perforations in their leaves throug

Q: Syncope during bathing in infants, a pediatric form of water-induced urticaria?
GOLD: yes   PRED: yes
Gold doc id: 9488747, retrieved: [9488747]
Model said: Yes

Explanation: The context mentions that the condition is described as a "pediatric form of water-induced urticaria," which aligns with syncope during bathing in infants being referred to as an exa


=== RIGHT DOC + WRONG ANSWER (LM reading fail) (39 cases, showing 2) ===
Q: Landolt C and snellen e acuity: differences in strabismus amblyopia?
GOLD: no   PRED: yes
Gold doc id: 16418930, retrieved: [16418

## Wrap-up: oral exam quick reference

**Most likely follow-ups:**

- **What is RAG and why use it?** Augments LM context with retrieved documents at inference time. Pros: factual grounding, citation, easy to update knowledge (just re-index). Cons: retrieval quality is a bottleneck; latency increases.
- **Why embed-then-retrieve instead of just keyword search?** Semantic similarity handles paraphrasing (`"cell death"` matches `"apoptosis"`) and synonyms that BM25 would miss. But for exact technical terms, BM25 is still competitive and is often combined with embeddings (hybrid retrieval).
- **Why chunking?** Embedders have context limits (MiniLM: 256 tokens). Long documents must be split. Chunk size trades precision (small) against coherence (large).
- **Why cosine similarity?** Magnitude of embeddings reflects "confidence" of the encoder, not semantic content. Cosine ignores magnitude, focuses on direction = pure semantic similarity. Also widely supported in vector stores.
- **What does `RunnableParallel` give you?** A way to compose two computations on the same input and merge their outputs into a dict. Lets us keep BOTH retrieved docs (for evaluation) AND the generated answer in one chain run.
- **When does RAG hurt instead of help?** Bad retrieval pulls in irrelevant text → LM gets distracted. Especially bad when the LM already knew the answer from pretraining. Visible in our experiments if `RAG accuracy < baseline accuracy`.
- **What's the difference between embedding the query vs the document?** Same encoder, but ideally a different prompt or strategy. Some embedders (like BGE) use special prefixes: `"Represent this sentence for retrieval: "` for queries.
